In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 datasets==2.19.0 evaluate bert_score sentencepiece

In [ ]:
import torch, gc, re, time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from peft import PeftModel
from datasets import load_dataset
from bert_score import score as bert_score_fn
import evaluate

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"

TRAIN_SIZE = 2000
VAL_SIZE   = 300
TEST_SIZE  = 200

MAX_CHUNK_WORDS    = 400
MIN_SECTION_WORDS  = 60
MATH_DENSITY_LIMIT = 0.12
CHUNK_MIN, CHUNK_MAX = 40, 90
FINAL_MIN, FINAL_MAX = 100, 200

device     = 0 if torch.cuda.is_available() else -1
DEVICE_STR = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'@xmath\d+',              '<equation>', text)
    text = re.sub(r'@xcite',                 '<citation>', text)
    text = re.sub(r'\[EQUATION\]',           '<equation>', text, flags=re.IGNORECASE)
    text = re.sub(r'\[EQ\.?\d*\]',          '<equation>', text, flags=re.IGNORECASE)
    text = re.sub(r'\\[a-zA-Z]+\{[^}]*\}',  '',           text)
    text = re.sub(r'\\[a-zA-Z]+',           '',           text)
    text = re.sub(r'\$[^$]{1,80}\$',        '<equation>', text)
    text = re.sub(r'\$\$[^$]+\$\$',         '<equation>', text)
    text = re.sub(r'[A-Z]_\{[^}]+\}',       '',           text)
    text = re.sub(r'[A-Za-z]+_[A-Za-z0-9]+(?=[^a-zA-Z]|$)', '', text)
    text = re.sub(r'[a-zA-Z]+\d*[a-zA-Z]*_[a-zA-Z]+\d*', '', text)
    text = re.sub(r'[A-Z][a-z]+(?:[A-Z][a-z]*){4,}', '', text)
    text = re.sub(r'[ \t]+',   ' ',    text)
    text = re.sub(r'\n{3,}',   '\n\n', text)
    return text.strip()

def split_sections(text: str) -> list:
    parts = text.split("\n\n")
    return [p.strip() for p in parts if len(p.split()) >= MIN_SECTION_WORDS]

def chunk_by_sentences(text: str, max_words: int = MAX_CHUNK_WORDS) -> list:
    sentences     = re.split(r'(?<=[.!?])\s+', text)
    chunks        = []
    current       = []
    current_words = 0
    for sent in sentences:
        n = len(sent.split())
        if current_words + n <= max_words:
            current.append(sent)
            current_words += n
        else:
            if current:
                chunks.append(" ".join(current))
            current, current_words = [sent], n
    if current:
        chunks.append(" ".join(current))
    return chunks

def is_math_heavy(chunk: str) -> bool:
    tokens = chunk.split()
    if not tokens:
        return False
    return (sum(1 for t in tokens if t == '<equation>') / len(tokens)) > MATH_DENSITY_LIMIT

def build_chunks(text: str) -> list:
    text     = clean_text(text)
    sections = split_sections(text)
    chunks   = []
    for section in sections:
        for chunk in chunk_by_sentences(section):
            if not is_math_heavy(chunk):
                chunks.append(chunk)
    return chunks

def run_two_pass_summary(summarizer, article_text: str) -> str:
    chunks = build_chunks(article_text)
    if not chunks:
        return "[No summarizable content found after cleaning.]"
    chunks = [c for c in chunks if len(c.split()) >= 50]
    if not chunks:
        return "[All chunks were too short to summarize.]"
    chunk_summaries = []
    for chunk in chunks:
        chunk_len = len(chunk.split())
        safe_min  = min(CHUNK_MIN, max(5,  chunk_len // 3))
        safe_max  = min(CHUNK_MAX, max(safe_min + 10, chunk_len))
        try:
            out = summarizer(chunk, min_length=safe_min, max_length=safe_max,
                             do_sample=False, truncation=True)
            chunk_summaries.append(out[0]["summary_text"])
        except Exception:
            continue
    if not chunk_summaries:
        return "[No chunks could be summarized.]"
    combined     = " ".join(chunk_summaries)
    final_chunks = chunk_by_sentences(combined, max_words=MAX_CHUNK_WORDS)
    final_chunks = [c for c in final_chunks if len(c.split()) >= 50]
    if not final_chunks:
        return combined
    final_parts = []
    for chunk in final_chunks:
        chunk_len = len(chunk.split())
        safe_min  = min(FINAL_MIN, max(10, chunk_len // 3))
        safe_max  = min(FINAL_MAX, max(safe_min + 10, chunk_len))
        try:
            out = summarizer(chunk, min_length=safe_min, max_length=safe_max,
                             do_sample=False, truncation=True)
            final_parts.append(out[0]["summary_text"])
        except Exception:
            continue
    return " ".join(final_parts) if final_parts else combined

In [ ]:
raw     = load_dataset("FlameF0X/arXiv-AI-ML", split="train").shuffle(seed=42)
test_ds = raw.select(range(TRAIN_SIZE + VAL_SIZE, TRAIN_SIZE + VAL_SIZE + TEST_SIZE))
print(f"Test set ready: {len(test_ds)} papers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Test set ready: 200 papers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_PATH = "/content/drive/MyDrive/bart-lora-adapter"

ft_base   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
ft_base.resize_token_embeddings(len(tokenizer))
ft_model  = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)
ft_model  = ft_model.merge_and_unload()
ft_model  = ft_model.to("cuda" if torch.cuda.is_available() else "cpu")

ft_summarizer = pipeline(
    "summarization",
    model=ft_model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=1024,
)
print("Model ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model ready


In [ ]:
! pip install rouge_score

In [ ]:
rouge_metric    = evaluate.load("rouge")
all_predictions = []
all_references  = []
print("Ready to evaluate")

Ready to evaluate


In [ ]:
SAMPLE_INDICES = [60,70]

for idx in SAMPLE_INDICES:
    torch.cuda.empty_cache()
    import gc; gc.collect()

    sample    = test_ds[idx]
    article   = sample["content"]
    reference = sample["abstract"]
    domain    = sample["domain"]
    title     = sample["title"]

    print(f"\n{'='*65}")
    print(f"TEST PAPER [{idx}] | Domain: {domain}")
    print(f"Title: {title}")
    print('='*65)

    chunks = build_chunks(article)
    print(f"Paper length  : {len(article.split())} words")
    print(f"Chunks (pass1): {len(chunks)}")

    print("\nGenerating summary...")
    t0      = time.time()
    summary = run_two_pass_summary(ft_summarizer, article)
    elapsed = round(time.time() - t0, 1)

    print(f"\n--- Reference (human abstract) ---")
    print(reference)
    print(f"\n--- Generated Summary ({elapsed}s) ---")
    print(summary)


TEST PAPER [60] | Domain: cs.CL
Title: ViMedCSS: A Vietnamese Medical Code-Switching Speech Dataset & Benchmark
Paper length  : 4999 words
Chunks (pass1): 16

Generating summary...

--- Reference (human abstract) ---
Code-switching (CS), which is when Vietnamese speech uses English words like drug names or procedures, is a common phenomenon in Vietnamese medical communication. This creates challenges for Automatic Speech Recognition (ASR) systems, especially in low-resource languages like Vietnamese. Current most ASR systems struggle to recognize correctly English medical terms within Vietnamese sentences, and no benchmark addresses this challenge. In this paper, we construct a 34-hour \textbf{Vi}etnamese \textbf{Med}ical \textbf{C}ode-\textbf{S}witching \textbf{S}peech dataset (ViMedCSS) containing 16,576 utterances. Each utterance includes at least one English medical term drawn from a curated bilingual lexicon covering five medical topics. Using this dataset, we evaluate several st

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
predictions = [summary]
references  = [reference]
print(f"Summary  : {len(summary.split())} words")
print(f"Reference: {len(reference.split())} words")

Summary  : 314 words
Reference: 147 words


In [ ]:
!pip install rouge_score

In [ ]:
import evaluate
from bert_score import score as bert_score_fn

rouge_metric = evaluate.load("rouge")
print("Evaluation metrics loaded ")

Evaluation metrics loaded 


In [ ]:
rouge_scores = rouge_metric.compute(
    predictions=predictions,
    references=references
)

print("── ROUGE Scores (fine-tuned BART, test set) ──")
print(f"  ROUGE-1 : {rouge_scores['rouge1']:.4f}")
print(f"  ROUGE-2 : {rouge_scores['rouge2']:.4f}")
print(f"  ROUGE-L : {rouge_scores['rougeL']:.4f}")
print()
print("Interpretation guide:")
print("  ROUGE-1 > 0.35 → good vocabulary overlap")
print("  ROUGE-2 > 0.10 → reasonable phrase-level match (hard for abstractive models)")
print("  ROUGE-L > 0.30 → decent structural alignment")

── ROUGE Scores (fine-tuned BART, test set) ──
  ROUGE-1 : 0.4389
  ROUGE-2 : 0.2911
  ROUGE-L : 0.3644

Interpretation guide:
  ROUGE-1 > 0.35 → good vocabulary overlap
  ROUGE-2 > 0.10 → reasonable phrase-level match (hard for abstractive models)
  ROUGE-L > 0.30 → decent structural alignment


In [ ]:
print("Computing BERTScore (this may take a few minutes)...")
P, R, F1 = bert_score_fn(
    predictions,
    references,
    lang="en",
    device ="cpu",
    verbose=False
)

print("── BERTScore (fine-tuned BART, test set) ──")
print(f"  Precision : {P.mean().item():.4f}")
print(f"  Recall    : {R.mean().item():.4f}")
print(f"  F1        : {F1.mean().item():.4f}  ← main number to report")
print()
print("Interpretation guide:")
print("  F1 > 0.85 → strong semantic match")
print("  F1 > 0.80 → acceptable for domain-specific abstractive summarization")
print("  F1 < 0.75 → model may be drifting from reference content")

Computing BERTScore (this may take a few minutes)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


── BERTScore (fine-tuned BART, test set) ──
  Precision : 0.8598
  Recall    : 0.9105
  F1        : 0.8844  ← main number to report

Interpretation guide:
  F1 > 0.85 → strong semantic match
  F1 > 0.80 → acceptable for domain-specific abstractive summarization
  F1 < 0.75 → model may be drifting from reference content
